In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from luh import AutoUncertaintyHead

from lm_polygraph import CausalLMWithUncertainty

from luh.calculator_infer_luh import CalculatorInferLuh
from luh.calculator_apply_uq_head import CalculatorApplyUQHead
from luh.luh_claim_estimator_dummy import LuhClaimEstimatorDummy

ImportError: cannot import name 'CausalLMWithUncertainty' from 'lm_polygraph' (/mnt/beegfs/work/xie12/uncertainty-guided-reasoning/UQ_Guided_Router/lm-polygraph/src/lm_polygraph/__init__.py)

In [2]:
from __future__ import annotations

import re
from typing import List, Tuple, Dict, Any

from tqdm import tqdm
from lm_polygraph.stat_calculators.stat_calculator import StatCalculator
from lm_polygraph.stat_calculators.extract_claims import Claim, WhiteboxModel


class StepsExtractor(StatCalculator):
    STEP_RE = re.compile(r'(?mi)(?:^|\n)(?P<marker>\s*-?\s*Step\s+\d+\s*:\s*)')
    ANSWER_RE = re.compile(r'(?mi)(?:^|\n)(?P<marker>\s*(?:<\s*Answer\s*>|Answer)\s*:\s*)')

    def __init__(
        self,
        skip_starts: List[str] | None = None,
        progress_bar: bool = True,
    ):
        super().__init__()
        self.skip_starts = skip_starts or ['Reasoning Steps:']
        self.progress_bar = progress_bar

    @staticmethod
    def meta_info() -> Tuple[List[str], List[str]]:
        return (
            [
                "claims",
                "claim_texts_concatenated",
                "claim_input_texts_concatenated",
            ],
            [
                "greedy_texts",
                "greedy_tokens",
            ],
        )

    def __call__(
        self,
        dependencies: Dict[str, object],
        texts: List[str],
        model: WhiteboxModel,
        max_new_tokens: int,
        *args,
        **kwargs,
    ) -> Dict[str, List]:
        claims: List[List[Claim]] = []
        claim_texts_concatenated: List[str] = []
        claim_input_texts_concatenated: List[str] = []

        data = zip(
            texts,
            dependencies["greedy_texts"],
            dependencies["greedy_tokens"],
        )
        if self.progress_bar:
            data = tqdm(data, total=len(texts), desc='Extracting steps')

        for input_text, greedy_text, greedy_tokens in data:
            steps: List[Claim] = self.split_to_steps(
                greedy_text, greedy_tokens, model.tokenizer
            )
            claims.append(steps)
            claim_texts_concatenated += [c.claim_text for c in steps]
            claim_input_texts_concatenated += [input_text for _ in steps]

        return {
            "claims": claims,
            "claim_texts_concatenated": claim_texts_concatenated,
            "claim_input_texts_concatenated": claim_input_texts_concatenated,
        }

    def filter_claim_texts(self, claim_text: str) -> bool:
        claim_text = claim_text.strip()
        return len(claim_text) > 0 and not any(
            claim_text.lower().startswith(b.lower()) for b in self.skip_starts
        )

    def _find_spans(self, text: str) -> List[Tuple[int, int]]:
        markers: List[int] = []

        for m in self.STEP_RE.finditer(text):
            markers.append(m.start("marker"))
        for m in self.ANSWER_RE.finditer(text):
            markers.append(m.start("marker"))

        if not markers:
            return []

        markers.sort()
        spans: List[Tuple[int, int]] = []
        for i, start in enumerate(markers):
            end = markers[i + 1] if i + 1 < len(markers) else len(text)
            spans.append((start, end))
        return spans

    def _char_to_token_index_boundaries(
        self, text: str, tokens: List[int], tokenizer, boundaries: List[int]
    ) -> List[int]:
        results: List[int] = []
        token_i = 0
        for boundary in sorted(boundaries):
            while token_i < len(tokens):
                next_decoded = tokenizer.decode(tokens[: token_i + 1])
                if next_decoded == text[: len(next_decoded)] and len(next_decoded) <= boundary:
                    token_i += 1
                else:
                    break
            results.append(token_i)
        return results

    def split_to_steps(
        self,
        text: str,
        tokens: List[int],
        tokenizer,
    ) -> List[Claim]:
        if not tokenizer.decode(tokens).startswith(text):
            return []

        spans = self._find_spans(text)
        if not spans:
            return []

        char_boundaries: List[int] = []
        for start, end in spans:
            char_boundaries.extend([start, end])

        token_boundaries = self._char_to_token_index_boundaries(
            text, tokens, tokenizer, char_boundaries
        )

        claims: List[Claim] = []
        for i, (start, end) in enumerate(spans):
            seg = text[start:end]
            if not self.filter_claim_texts(seg):
                continue
            tok_start = token_boundaries[2 * i]
            tok_end = token_boundaries[2 * i + 1]
            aligned_ids = list(range(tok_start, min(tok_end, max(len(tokens) - 1, 0))))

            claims.append(
                Claim(
                    claim_text=seg.strip(),
                    sentence=seg,
                    aligned_token_ids=aligned_ids,
                )
            )

        return claims


def load_stat_calculator(config, builder):
    return StepsExtractor(
        progress_bar=getattr(config, "progress_bar", False),
    )
    

In [3]:
# load model and uhead
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# uhead_name = "llm-uncertainty-head/uhead_Mistral-7B-Instruct-v0.2"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
uhead_name = "rediska0123/uhead_Qwen2.5-1.5B-Instruct_6epochs"


llm = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(
    model_name)
tokenizer.pad_token = tokenizer.eos_token
uhead = AutoUncertaintyHead.from_pretrained(
    uhead_name, base_model=llm)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [ ]:
generation_config = GenerationConfig.from_pretrained(model_name)
args_generate = {"generation_config": generation_config, "max_new_tokens": 200}
calc_infer_llm = CalculatorInferLuh(
    uhead,
    tokenize=True,
    args_generate=args_generate,
    device="cuda",
    generations_cache_dir="",
    predict_token_uncertainties=False,
)

calc_steps_extractor = StepsExtractor()
calc_apply_uhead = CalculatorApplyUQHead(uhead)

estimator = LuhClaimEstimatorDummy()
llm_adapter = CausalLMWithUncertainty(
    llm,
    tokenizer=tokenizer,
    stat_calculators=[calc_infer_llm, calc_steps_extractor, calc_apply_uhead],
    estimator=estimator,
)

In [ ]:

# prepare text ...

messages = [
    [
        {
            "role": "system",
            "content": "You are an expert mathematical reasoner.",
        },
        {
            "role": "user", 
            "content": "You are given the following problem:\nWhat is the sum of 123 and 456?\nReason step by step: Step 1. \n Step 2. \n Step 3. \n"
            #"content": "How to compute a square root of a number? Reasong step by step. Step 1. \n Step 2. \n"
            #"content": "In which year did the programming language Mercury first appear? Reasong step by step. Step 1. \n Step 2. \n"
        }
    ]
]
# The correct answer is 1995

chat_messages = [tokenizer.apply_chat_template(m, tokenize=False, add_bos_token=False) for m in messages]
inputs = tokenizer(chat_messages, return_tensors="pt", padding=True, truncation=True, add_special_tokens=False).to("cuda")

output = llm_adapter.generate(inputs["input_ids"], max_new_tokens=200)
output["uncertainty_score"]

/home/artemshelmanov/conda/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/artemshelmanov/conda/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/artemshelmanov/conda/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


odict_keys(['sequences', 'scores', 'attentions', 'hidden_states', 'past_key_values', 'full_attention_mask', 'context_lengths'])


Extracting steps: 100%|██████████| 1/1 [00:00<00:00, 310.74it/s]
/home/artemshelmanov/conda/lib/python3.12/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[[0.03869924634153558, 0.41271158703675315, 0.7928948484199027]]

In [9]:
print("Model response and uncertainty scores:")
print(f'Response: {tokenizer.batch_decode(output["sequences"][:,len(inputs["input_ids"][0]):])}')
print(f'UE Scores: {output["uncertainty_score"][0]}')

Model response and uncertainty scores:
Response: ['<|im_start|>system\nTo find the sum of 123 and 456, we can follow these steps:\n\nStep 1: Align the numbers vertically by their place values.\n```\n   123\n+ 456\n------\n```\n\nStep 2: Add the digits in each column from right to left.\n\n- In the ones place: 3 + 6 = 9\n- In the tens place: 2 + 5 = 7\n- In the hundreds place: 1 + 4 = 5\n\nStep 3: Write down the sums for each column, starting from the bottom up.\n\n```\n   123\n+ 456\n------\n  579\n```\n\nTherefore, the sum of 123 and 456 is 579.<|im_end|>']
UE Scores: [0.03869924634153558, 0.41271158703675315, 0.7928948484199027]


In [6]:
def highlight_html_tokens(
    token_ids,
    positions_to_highlight,
    tokenizer,
    color="red",
    font_weight="bold"
):
    """
    Convert a list of token IDs into a readable string, highlight tokens at
    the specified positions in `positions_to_highlight`, and remove the leading
    '▁' that Mistral/Llama tokenizers use for word boundaries.
    
    Args:
        token_ids (List[int]): The sequence of token IDs.
        tokenizer: A Hugging Face tokenizer (e.g., for mistralai/Mistral-7B-Instruct-v0.2).
        positions_to_highlight (Set[int] or List[int]): 0-based indices of tokens to highlight.
        color (str): CSS color for the highlighted text (default "red").
        font_weight (str): CSS font weight (default "bold").
    
    Returns:
        str: An HTML string with some tokens highlighted.
    """
    # Convert the IDs to subword tokens (may contain leading "▁")
    raw_tokens = tokenizer.convert_ids_to_tokens(token_ids)
    
    # Ensure positions_to_highlight is a set for quick membership check
    if not isinstance(positions_to_highlight, set):
        positions_to_highlight = set(positions_to_highlight)
    
    final_pieces = []
    
    for idx, token in enumerate(raw_tokens):
        # If the token starts with "▁", replace that with a literal space
        if token.startswith("▁"):
            display_str = " " + token[1:]
        else:
            display_str = token
        
        # If this position is in positions_to_highlight, wrap in <span>
        if idx in positions_to_highlight:
            display_str = (
                f"<span style='color:{color}; font-weight:{font_weight};'>"
                f"{display_str}"
                "</span>"
            )
        
        final_pieces.append(display_str)
    
    # Join everything without extra spaces
    return "".join(final_pieces)

In [7]:
from IPython.display import HTML


def highlight_uncertain_claims(uncertainties, generated_tokens, claims):
    threshold = 0.5
    tokens_to_highlight = set()

    for ue_score, claim in zip(uncertainties, claims):
        if ue_score > threshold:
            tokens_to_highlight.update([claim])
    
    display(HTML(highlight_html_tokens(generated_tokens, tokens_to_highlight, tokenizer)))

In [8]:
highlight_uncertain_claims(
    output["uncertainty_score"][0],
    output["sequences"][:,len(inputs["input_ids"][0]):][0],
    list(range(len(output["sequences"][:,len(inputs["input_ids"][0]):][0]))),
)